## 6. Dunder (Magic) Methods

**Dunder methods** (double underscore, e.g. `__str__`) are special methods that Python calls **automatically** in response to built-in operations. They let your objects behave like built-in types.

### Most important dunder methods
| Method | Called when | Example trigger |
|--------|-------------|----------------|
| `__init__` | Object created | `MyClass()` |
| `__repr__` | `repr(obj)` / in REPL | `repr(obj)` |
| `__str__` | `str(obj)` / `print(obj)` | `print(obj)` |
| `__len__` | `len(obj)` | `len(obj)` |
| `__getitem__` | `obj[key]` | `obj[0]` |
| `__setitem__` | `obj[key] = val` | `obj[0] = 5` |
| `__contains__` | `item in obj` | `5 in obj` |
| `__iter__` | `for x in obj` | `for x in obj` |
| `__add__` | `obj + other` | `obj + obj2` |
| `__eq__` | `obj == other` | `obj == obj2` |
| `__lt__` | `obj < other` | `obj < obj2` |
| `__bool__` | `bool(obj)` / `if obj:` | `if obj:` |
| `__call__` | `obj()` | `obj()` |
| `__enter__`/`__exit__` | `with obj as x:` | context manager |

> 💡 **`__repr__`** should return an unambiguous string that could recreate the object: `ClassName(arg1, arg2)`. **`__str__`** should return a human-friendly string. If only `__repr__` is defined, it is used for `str()` too.

In [2]:
import math
from functools import total_ordering

# ---- Vector class with rich dunder support ----
@total_ordering   # auto-generates >, >=, <= from __eq__ and __lt__
class Vector:
    """2-D mathematical vector."""

    def __init__(self, x, y):
        self.x = x
        self.y = y

    # String representations
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"    # unambiguous

    def __str__(self):
        return f"({self.x}i + {self.y}j)"        # friendly

    # Arithmetic
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):    # scalar * vector
        return self.__mul__(scalar)

    def __neg__(self):
        return Vector(-self.x, -self.y)

    # Magnitude as len
    def __len__(self):
        return int(math.hypot(self.x, self.y))

    def magnitude(self):
        return math.hypot(self.x, self.y)

    # Comparison
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __lt__(self, other):
        return self.magnitude() < other.magnitude()

    # Boolean
    def __bool__(self):
        return self.x != 0 or self.y != 0


v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(f"repr: {repr(v1)}")
print(f"str:  {str(v1)}")
print(f"print: ", v1)          # calls __str__
print(f"v1 + v2 = {v1 + v2}")
print(f"v1 - v2 = {v1 - v2}")
print(f"v1 * 3  = {v1 * 3}")
print(f"3 * v1  = {3 * v1}")
print(f"-v1     = {-v1}")
print(f"len(v1) = {len(v1)}   (int magnitude)")
print(f"v1 == v1: {v1 == v1}")
print(f"v1 == v2: {v1 == v2}")
print(f"v1 > v2:  {v1 > v2}")
print(f"bool(v1): {bool(v1)}")
print(f"bool(Vector(0,0)): {bool(Vector(0,0))}")
print(f"sorted:   {sorted([v1, v2, Vector(5,5)])}")

repr: Vector(3, 4)
str:  (3i + 4j)
print:  (3i + 4j)
v1 + v2 = (4i + 6j)
v1 - v2 = (2i + 2j)
v1 * 3  = (9i + 12j)
3 * v1  = (9i + 12j)
-v1     = (-3i + -4j)
len(v1) = 5   (int magnitude)
v1 == v1: True
v1 == v2: False
v1 > v2:  True
bool(v1): True
bool(Vector(0,0)): False
sorted:   [Vector(1, 2), Vector(3, 4), Vector(5, 5)]


In [3]:
# ---- Context manager with __enter__ / __exit__ ----
class Timer:
    """Measures elapsed time using 'with' block."""
    import time

    def __enter__(self):
        import time
        self._start = time.perf_counter()
        print("  Timer started...")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        import time
        self.elapsed = time.perf_counter() - self._start
        print(f"  Timer stopped: {self.elapsed:.4f}s")
        return False   # don't suppress exceptions

with Timer() as t:
    total = sum(i**2 for i in range(100_000))
print(f"  sum = {total:,}")

  Timer started...
  Timer stopped: 0.0267s
  sum = 333,328,333,350,000


In [4]:
# ---- __call__ — callable object ----
class Multiplier:
    """An object that behaves like a function."""
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor

double = Multiplier(2)
triple = Multiplier(3)
print(f"double(7) = {double(7)}")
print(f"triple(7) = {triple(7)}")
print(f"callable(double): {callable(double)}")

double(7) = 14
triple(7) = 21
callable(double): True
